In [0]:
#%run ./load_data ----- A décommmenter pour lancer les notebooks séparements

In [0]:
#from functools import reduce

#DM-5912
df_measures_raw = reduce(
    lambda a, b: a.union(b),
    [
        mesures_calculees_ro1,
        mesures_calculees_ng1,
        mesures_calculees_ng2,
        mesures_calculees_pr1,
        mesures_calculees_st1,
        mesures_calculees_st2,
        mesures_calculees_po1,
        mesures_calculees_bu1, 
        mesures_calculees_bo1
    ]
)

# Filtre supprimé : déjà appliqué dans load_data (mesures_filter)
df_measures = df_measures_raw.select(
    F.col("prd_line"),
    F.coalesce(F.col("batch_id"), F.col("mes_number")).alias("batch_id"),
    F.col("measure_name").alias("mesure_name"),
    F.col("start").alias("date_debut"),
    F.col("end").alias("date_fin"),
    F.col("measure.value").alias("mesure_value"),
    F.col("measure.computed_value").alias("computed_value"),
    F.col("created_at"),
    F.col("modified_at")
)

In [0]:
#DM-5912
#DM-5912
mapping = {
    "nogent1": "NG1",
    "nogent2": "NG2",
    "rouen1": "RO1",
    "prouvy1": "PR1",
    "strasbourg1": "ST1",
    "strasbourg2": "ST2",
    "polisy1": "PO1",
    "buzau1": "BU1", 
    "bolelemi1": "BO1"
}

mapping_expr = F.create_map([F.lit(x) for x in sum(mapping.items(), ())])

df_metadata = (
    metadata_mesures
    .withColumn(
        "prd_line",
        F.coalesce(mapping_expr[F.lower(F.col("prd_line"))], F.col("prd_line"))
    )
    .withColumn("Colonne_1", F.expr("get(columns, 0)"))
    .withColumn("Colonne_2", F.expr("get(columns, 1)"))
    .withColumn("mesure_name", F.lower(F.col("measure_DE")))
)


In [0]:
# (ajout prd_line)
ascendance = reduce(
    lambda a, b: a.union(b),
    [ascendance_ro1, ascendance_ng1, ascendance_ng2, ascendance_pr1, 
     ascendance_st1, ascendance_st2, ascendance_po1, ascendance_bu1, ascendance_bo1]
)

In [0]:
localisation = localisation.filter(F.col("batch_id").isNotNull())

#measurement = measurement.filter(F.col("batch_id").isNotNull())

In [0]:
# Mapping mois en français
mois_fr = {
    1: "janvier", 2: "février", 3: "mars", 4: "avril",
    5: "mai", 6: "juin", 7: "juillet", 8: "août",
    9: "septembre", 10: "octobre", 11: "novembre", 12: "décembre"
}

# Optimisé : create_map natif Spark au lieu d'une UDF Python (évite la sérialisation)
mois_fr_map = F.create_map([F.lit(x) for x in sum(mois_fr.items(), ())])

def add_date_columns(df, date_col="date_fin_prod"):
    return (
        df
        .withColumn("week", F.weekofyear(date_col))
        .withColumn("year", F.year(date_col))
        .withColumn("month_num", F.month(date_col))
        .withColumn("month_name", mois_fr_map[F.col("month_num")])
        .withColumn("month_label", F.concat_ws(".", F.col("month_num"), F.col("month_name")))
        .drop("month_num")
    )


In [0]:
# Optimisé : broadcast sur les petites tables de dimension
batches_info = batches.alias("a").join(
    F.broadcast(plants_production_lines).alias("b"),
    F.col("a.production_line") == F.col("b.id_plant_production_line"),
    "left"
).join(
    F.broadcast(varieties).alias("c"),
    F.col("a.variety") == F.col("c.id_good_variety"),
    "left").join(
        F.broadcast(species).alias("d"),
        F.col("c.specy") == F.col("d.id_good_specy"),
        "left"
    ).join(
    F.broadcast(requirement_specifications).alias("e"),
    F.col("a.requirement_specifications") == F.col("e.id_requirement_specification"),
    "left"
    ).join(
        manual_entries.alias("f"),
        F.col("a.id_batch") == F.col("f.batch"),
        "left"
    ).join(
        F.broadcast(parameters_variables).alias("g"),
        F.col("f.parameter") == F.col("g.id_parameter_variable"),


    "left"
    ).join(
        F.broadcast(parameters_cycles).alias("pc"),
        F.col("a.batch_cycle") == F.col("pc.id_parameter_batch_cycle"),


        "left"
    ).join(
        F.broadcast(production_type).alias("h"),
        F.col("a.production_type") == F.col("h.id_parameter_production_type"),
        "left"
    ).filter(
       (F.col("g.code") == "goods_weight") &
       (F.col("f.deleted") == False) &

       (F.col("pc.deleted") == False) &

       (F.col("a.deleted") == False) ).select(
    F.col("a.id_batch").alias("batch_id"),
    F.col("a.batch_number"),
    F.col("a.mes_number"),
    F.col("b.name").alias("production_line"),
    F.col("b.id_plant_production_line"),
    F.col("h.code").alias("production_type"),
    F.col("e.name").alias("requirement_specifications"),
    F.col("d.code").alias("specy_name"),
    F.col("c.code").alias("variety_name"),
    # --- Phase 2 : identifiants requis pour joindre les tables de traduction.
    # Les colonnes *_name ci-dessus sont des codes métier, pas des clés : elles
    # ne permettent aucune jointure vers les tables *_translations, qui portent
    # toutes une clé entière. On les conserve (tri, compatibilité de l'existant)
    # et on ajoute les identifiants à côté.
    F.col("d.id_good_specy"),
    F.col("c.id_good_variety"),
    F.col("h.id_parameter_production_type"),
    F.col("e.id_requirement_specification"),
    F.col("f.value").alias("goods_weight"),

    F.col("pc.cycle_duration").alias("cycle_duration"),
    F.col("pc.cycle_label").alias("cycle_label"),
    
    F.col("a.planned_date").alias("planned_date"),
    F.col("a.planned_datetime").alias("planned_datetime")
    )

batches_info = batches_info.filter(F.col("production_line").isin("ROUEN1", "NOGENT1", "NOGENT2", "PROUVY1", "STRASBOURG1", "STRASBOURG2", "POLISY1", "BUZAU1", "BOLELEMI1")) # (ajout prd_line)
#batches_info = batches_info.filter(F.col("planned_datetime") >= "2023-01-01")

In [0]:
batches_status = batches_info.alias("a").join(
    localisation.alias("b"),
    F.col("a.batch_id") == F.col("b.batch_id"),
    "left"
).select(
    "a.*",
    F.col("b.batch_status").alias("batch_status_localization"),
    "b.error_messages")

In [0]:
prd_lines = ['rouen1', 'nogent2', 'nogent1','prouvy1', 'strasbourg1', 'strasbourg2', 'polisy1', 'buzau1', 'bolelemi1'] # (ajout prd_line)
tables = {}

for line in prd_lines:
    catalog_mal_maite_gold = f"mal_maite_{line}_{current_environment}.gold"
    table_name = f"{catalog_mal_maite_gold}.localization_events"
    tables[line] = spark.table(table_name)




localization_events = reduce(lambda df1, df2: df1.unionByName(df2), tables.values())

localization_events = localization_events.select("prd_line", "prd_workshop", "prd_cell", "batch_id").dropDuplicates()

localization_events = localization_events.withColumn(
    "steep_vessel1",
    F.when((F.col("prd_workshop") == "steeping") & (F.col("prd_cell") == 1), "steep_vessel1").otherwise(F.lit(None))
).withColumn(
    "steep_vessel2",
    F.when((F.col("prd_workshop") == "steeping") & (F.col("prd_cell") == 2), "steep_vessel2").otherwise(F.lit(None))
).withColumn(
    "germ_vessel1",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 1), "germ_vessel1").otherwise(F.lit(None))
).withColumn(
    "germ_vessel2",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 2), "germ_vessel2").otherwise(F.lit(None))
).withColumn(
    "germ_vessel3",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 3), "germ_vessel3").otherwise(F.lit(None))
).withColumn(
    "germ_vessel4",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 4), "germ_vessel4").otherwise(F.lit(None))
).withColumn(
    "germ_vessel5",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 5), "germ_vessel5").otherwise(F.lit(None))
).withColumn(
    "germ_vessel6",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 6), "kgerm_vessel6").otherwise(F.lit(None))
).withColumn(
    "kiln_vessel1",
    F.when((F.col("prd_workshop") == "kilning") & (F.col("prd_cell") == 1), "kiln_vessel1").otherwise(F.lit(None))
).withColumn(
    "kiln_vessel2",
    F.when((F.col("prd_workshop") == "kilning") & (F.col("prd_cell") == 2), "kiln_vessel2").otherwise(F.lit(None))
)



# Agréger chaque colonne par batch_id en prenant la première valeur non nulle
df_aggregated = localization_events.groupBy("batch_id").agg(
    first("steep_vessel1", ignorenulls=True).alias("steep_vessel1"),
    first("steep_vessel2", ignorenulls=True).alias("steep_vessel2"),
    first("germ_vessel1", ignorenulls=True).alias("germ_vessel1"),
    first("germ_vessel2", ignorenulls=True).alias("germ_vessel2"),
    first("germ_vessel3", ignorenulls=True).alias("germ_vessel3"),
    first("germ_vessel4", ignorenulls=True).alias("germ_vessel4"),
    first("germ_vessel5", ignorenulls=True).alias("germ_vessel5"),
    first("germ_vessel6", ignorenulls=True).alias("germ_vessel6"),
    first("kiln_vessel1", ignorenulls=True).alias("kiln_vessel1"),
    first("kiln_vessel2", ignorenulls=True).alias("kiln_vessel2")
)

df_aggregated = df_aggregated.alias("a").join(
    batches.alias("b"), 
    F.col("a.batch_id") == F.col("b.id_batch"), 
    "left"
).select(
    "b.batch_number",
    F.col("b.planned_datetime").alias("planned_date"),
    "a.*"
)

localization_events = df_aggregated

####Evol

df_mesures_all_avec_cuves = df_aggregated.select(
    "batch_id",
    "steep_vessel1",
    "steep_vessel2",
    "germ_vessel1",
    "germ_vessel2",
    "germ_vessel3",
    "germ_vessel4",
    "germ_vessel5",
    "germ_vessel6",
    "kiln_vessel1",
    "kiln_vessel2"
)

old_stak_localisation_filtered = old_stak_localisation_filtered.unionByName(localization_events)